<!-- mccl/notebooks/continual_benchmark_waste5.ipynb -->
# Benchmark d'apprentissage continu — 5 classes de déchets

**Protocole :**
1. 5 classes : Organique, Textile, Verre, Plastique, Canette
2. **Tâche 0 = 2 classes**
3. **Tâche 1 = 2 classes**
4. **Tâche 2 = 1 classe**
5. Trois stratégies comparées : Replay, EWC, DualMemory
6. Détection du point de rupture et analyse des faiblesses

## 1. Setup

In [7]:
import sys
from pathlib import Path
import copy
import pickle
import zipfile
import time

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    import src.outils.data_loader  # noqa: F401
except ImportError as e:
    print("outils.data_loader :", e)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image

from src.config import ModelCfg, ManifoldCfg, RegCfg
from src.models.jepa import JepaModel
from src.training.optim import build_optimizer
from src.training.losses import classification_loss
from src.training.trainer import train_one_task
from src.evaluation.metrics import per_class_accuracy
from src.continual.strategies import build_strategy
from src.utils.seed import set_seed
from src.utils.logging import get_logger

set_seed(0)
log = get_logger("bench")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("ROOT   :", ROOT)
print("device :", device)
print("torch  :", torch.__version__)

ROOT   : c:\Users\hp\Documents\mccl
device : cpu
torch  : 2.14.1+cpu


## 2. Chargement des 5 classes de déchets

In [8]:
def load_zip(zip_path, internal="loader.pkl"):
    with zipfile.ZipFile(zip_path, "r") as zf:
        with zf.open(internal) as f:
            l = pickle.load(f, encoding="latin1")
    if l.data_ is None:
        l.data_ = copy.deepcopy(l.original_data_)
    return l


ZIP_PATH = ROOT / "data" / "dataset2.zip"
if not ZIP_PATH.exists():
    ZIP_PATH = ROOT / "dataset2.zip"

loader = load_zip(ZIP_PATH)
real_df = loader.data_
print(f"Shape   : {real_df.shape}")
print(f"Classes : {loader.name_label_}")

# Le dataset contient déjà 5 classes
WASTE_CLASSES = ["Organique", "Textile", "Verre", "Plastique", "Canette"]

# Vérifier que les 5 classes sont présentes
present = set(loader.name_label_)
missing = [c for c in WASTE_CLASSES if c not in present]
if missing:
    raise ValueError(f"Classes manquantes : {missing}")

# Mapping ordre alphabétique
selected = sorted(WASTE_CLASSES)
class_mapping = {c: i for i, c in enumerate(selected)}
name_label = selected

# Filtrer
mask = real_df["Label"].isin(selected)
real_df = real_df[mask].reset_index(drop=True)

# Remapper les labels
real_df["Label"] = real_df["Label"].map(class_mapping)

# Mettre à jour le loader
loader.data_ = real_df
loader.class_mapping_ = class_mapping
loader.name_label_ = name_label

n_classes = len(name_label)
print(f"\nAprès filtrage :")
print(f"  Shape   : {real_df.shape}")
print(f"  Classes : {n_classes} → {name_label}")
print(f"  Mapping : {class_mapping}")
print(f"\nRépartition :")
print(real_df["Label"].value_counts().sort_index())

assert n_classes == 5, f"Attendu 5, trouvé {n_classes}"

Shape   : (12138, 2)
Classes : ['Canette', 'Organique', 'Plastique', 'Textile', 'Verre']

Après filtrage :
  Shape   : (12138, 2)
  Classes : 5 → ['Canette', 'Organique', 'Plastique', 'Textile', 'Verre']
  Mapping : {'Canette': 0, 'Organique': 1, 'Plastique': 2, 'Textile': 3, 'Verre': 4}

Répartition :
Label
0    1792
1    2342
2    2583
3    2665
4    2756
Name: count, dtype: int64


## 3. Split 70/15/15

In [9]:
from sklearn.model_selection import train_test_split

labels = real_df["Label"].to_numpy(dtype=int)
idx = np.arange(len(real_df))

tr, tmp = train_test_split(idx, test_size=0.30, stratify=labels, random_state=0)
va, te = train_test_split(tmp, test_size=0.50, stratify=labels[tmp], random_state=0)

print(f"train={len(tr)} | val={len(va)} | test={len(te)}")
print(f"Classes par split (train) : min={np.bincount(labels[tr]).min()}, max={np.bincount(labels[tr]).max()}")

train=8496 | val=1821 | test=1821
Classes par split (train) : min=1254, max=1929


## 4. Dataset + transforms

In [10]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

transform_train = T.Compose([
    T.RandomResizedCrop(224, scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.3, 0.3, 0.3, 0.05),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

transform_eval = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class ImageSubset(Dataset):
    def __init__(self, images, labels, indices, transform=None):
        self.images = images
        self.labels = labels
        self.indices = np.asarray(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        j = int(self.indices[i])
        img = self.images[j]

        if not isinstance(img, Image.Image):
            arr = np.asarray(img)
            if arr.ndim == 3 and arr.shape[0] in (1, 3) and arr.shape[0] != arr.shape[-1]:
                arr = np.transpose(arr, (1, 2, 0))
            if arr.dtype != np.uint8:
                arr = (arr * 255).clip(0, 255).astype(np.uint8) if arr.max() <= 1.0 \
                      else arr.clip(0, 255).astype(np.uint8)
            if arr.ndim == 3 and arr.shape[2] == 1:
                arr = arr[:, :, 0]
            img = Image.fromarray(arr)

        if img.mode != "RGB":
            img = img.convert("RGB")

        if self.transform is not None:
            img = self.transform(img)
        else:
            img = T.ToTensor()(img)

        return img, int(self.labels[j])


images = real_df["Image"].tolist()

## 5. Ordre des tâches : 2 → 2 → 1

Avec 5 classes :
- **Tâche 0** : classes {a, b} (2 classes)
- **Tâche 1** : classes {a, b, c, d} (2 nouvelles)
- **Tâche 2** : classes {a, b, c, d, e} (1 nouvelle)

In [11]:
# Protocole : 2 classes → 2 classes → 1 classe
TASK_SIZES = [2, 2, 1]

rng = np.random.default_rng(42)
class_order = rng.permutation(n_classes).tolist()
print("Ordre des classes :", class_order)
print("Noms             :", [name_label[c] for c in class_order])

tasks = []
consumed = 0
for task_idx, size in enumerate(TASK_SIZES):
    new_classes = class_order[consumed:consumed + size]
    consumed += len(new_classes)
    tasks.append({
        "idx": task_idx,
        "classes": new_classes,
        "seen": class_order[:consumed],
    })

print(f"\n{len(tasks)} tâches :")
for t in tasks:
    names_new = [name_label[c] for c in t["classes"]]
    print(f"  Tâche {t['idx']} : new={t['classes']} ({names_new}), total_vues={len(t['seen'])}")

Ordre des classes : [4, 2, 3, 1, 0]
Noms             : ['Verre', 'Plastique', 'Textile', 'Organique', 'Canette']

3 tâches :
  Tâche 0 : new=[4, 2] (['Verre', 'Plastique']), total_vues=2
  Tâche 1 : new=[3, 1] (['Textile', 'Organique']), total_vues=4
  Tâche 2 : new=[0] (['Canette']), total_vues=5


## 6. Fonctions d'entraînement et d'évaluation

In [14]:
def make_cfg_euclidean():
    return ModelCfg(
        name="euclid",
        manifold=ManifoldCfg(
            type="product",
            components=[{"type": "euclidean", "dim": 192}],
            pooling="mean",
        ),
        reg=RegCfg(lambda_h=0.0, lambda_s=0.0, lambda_e=0.0),
    )


@torch.no_grad()
def eval_per_class(model, loader, class_ids):
    model.eval()
    id_to_local = {cid: i for i, cid in enumerate(class_ids)}

    logits_all, y_all = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        out = model(xb)
        if out.logits is not None:
            logits_all.append(out.logits.cpu())
            y_all.append(yb)

    if not logits_all:
        return {}

    logits = torch.cat(logits_all, 0)
    y = torch.cat(y_all, 0)
    y_local = torch.tensor(
        [id_to_local.get(int(v), -1) for v in y],
        dtype=torch.long,
    )
    valid = y_local >= 0
    if valid.sum() == 0:
        return {}

    pc = per_class_accuracy(logits[valid], y_local[valid], len(class_ids))
    return {
        class_ids[i]: float(pc[i])
        for i in range(len(class_ids))
        if not np.isnan(pc[i])
    }


def run_strategy(strategy_name, n_epochs=10, batch_size=32, lr=1e-3, seed=0):
    set_seed(seed)
    model = JepaModel(make_cfg_euclidean()).to(device)
    opt = build_optimizer(model, lr=lr)

    if strategy_name == "replay":
        strategy = build_strategy("replay", capacity=500)
    elif strategy_name == "ewc":
        strategy = build_strategy("ewc", lam=1e4, n_samples=256)
        strategy.attach_model(model)
    elif strategy_name == "dual_memory":
        strategy = build_strategy(
            "dual_memory",
            fast_cfg={"capacity": 500},
            slow_cfg={"capacity_per_class": 32},
        )
    else:
        raise ValueError(strategy_name)

    T = len(tasks)
    acc_matrix = np.full((T, n_classes), np.nan)
    t_start = time.time()

    for task in tasks:
        t_idx = task["idx"]
        new_classes = task["classes"]
        model.register_new_classes(new_classes)
        class_ids = model.classifier.class_ids()
        id_to_local = {cid: i for i, cid in enumerate(class_ids)}

        strategy.begin_task(type("T", (), {"index": t_idx, "classes": new_classes})())

        train_mask = np.isin(labels[tr], new_classes)
        ds_t = ImageSubset(images, labels, tr[train_mask], transform_train)
        dl_t = DataLoader(ds_t, batch_size=batch_size, shuffle=True, num_workers=0)

        def loss_fn(m, batch, out):
            xb, yb = batch
            yb = yb.to(device)
            yb_local = torch.tensor(
                [id_to_local[int(v)] for v in yb.cpu()],
                device=device, dtype=torch.long,
            )
            loss = classification_loss(out.logits, yb_local)

            replay = strategy.sample_replay(xb.size(0))
            if replay is not None:
                xr, yr = replay
                xr = xr.to(device)
                yr = yr.to(device)
                out_r = m(xr)
                yr_local = torch.tensor(
                    [id_to_local.get(int(v), -1) for v in yr.cpu()],
                    device=device, dtype=torch.long,
                )
                valid_r = yr_local >= 0
                if valid_r.sum() > 0:
                    loss = loss + classification_loss(
                        out_r.logits[valid_r], yr_local[valid_r]
                    )

            pen = strategy.penalty()
            if pen.requires_grad or pen.item() > 0:
                loss = loss + pen
            return loss

        def on_step(state, batch):
            xb, yb = batch
            with torch.no_grad():
                out = model(xb.to(device))
            strategy.observe((xb.detach().cpu(), yb.detach().cpu()), out=out)

        train_one_task(
            model=model, train_loader=dl_t, optimizer=opt,
            n_epochs=n_epochs, loss_fn=loss_fn, device=device,
            on_step=on_step, task_idx=t_idx, verbose=True,
        )

        strategy.end_task(
            type("T", (), {"index": t_idx, "classes": new_classes})(),
            train_loader=dl_t,
        )

        test_mask = np.isin(labels[te], task["seen"])
        ds_test = ImageSubset(images, labels, te[test_mask], transform_eval)
        dl_test = DataLoader(ds_test, batch_size=64, shuffle=False, num_workers=0)

        per_class = eval_per_class(model, dl_test, class_ids)
        for c, acc in per_class.items():
            acc_matrix[t_idx, c] = acc

        new_accs = [per_class[c] for c in new_classes if c in per_class]
        new_acc = float(np.mean(new_accs)) if new_accs else 0.0

        old_classes = [c for c in task["seen"] if c not in new_classes]
        old_accs = [per_class[c] for c in old_classes if c in per_class]
        old_acc = float(np.mean(old_accs)) if old_accs else float("nan")

        all_accs = [per_class[c] for c in task["seen"] if c in per_class]
        overall = float(np.mean(all_accs)) if all_accs else 0.0

        print(
            f"  Tâche {t_idx} | new={[name_label[c] for c in new_classes]} | "
            f"new_acc={new_acc:.3f} | old_acc={old_acc:.3f} | "
            f"overall={overall:.3f} | t={time.time()-t_start:.0f}s"
        )

    return model, strategy, acc_matrix, time.time() - t_start

## 7. Lancer les 3 stratégies avec sauvegarde

In [ ]:
# --- Checkpoints ---
CKPT_DIR = ROOT / "results" / "checkpoints_waste5"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
STATE_FILE = CKPT_DIR / "_state.pkl"
CSV_FILE = ROOT / "results" / "benchmark_waste5.csv"

STRATEGIES = ["ewc", "replay", "dual_memory"]


def get_strategy_state(strategy):
    if not hasattr(strategy, "state"):
        return {}
    s = strategy.state
    if callable(s):
        try:
            return s()
        except Exception:
            return {}
    if hasattr(s, "__dict__"):
        return dict(s.__dict__)
    if isinstance(s, dict):
        return s
    return {"value": str(s)}


def save_checkpoint(strat, acc_matrix, duration, strategy_state):
    ckpt_file = CKPT_DIR / f"{strat}.pkl"
    payload = {
        "strat": strat,
        "acc_matrix": acc_matrix,
        "duration": duration,
        "strategy_state": strategy_state,
        "timestamp": time.time(),
    }
    tmp = ckpt_file.with_suffix(".pkl.tmp")
    with open(tmp, "wb") as f:
        pickle.dump(payload, f)
    tmp.replace(ckpt_file)

    state = load_state()
    state[strat] = {"done": True, "duration": duration,
                    "timestamp": payload["timestamp"]}
    tmp_state = STATE_FILE.with_suffix(".pkl.tmp")
    with open(tmp_state, "wb") as f:
        pickle.dump(state, f)
    tmp_state.replace(STATE_FILE)

    update_csv()
    print(f"  💾 Sauvegardé : {ckpt_file.name}")


def load_state():
    if STATE_FILE.exists():
        with open(STATE_FILE, "rb") as f:
            return pickle.load(f)
    return {}


def load_checkpoint(strat):
    ckpt_file = CKPT_DIR / f"{strat}.pkl"
    if ckpt_file.exists():
        with open(ckpt_file, "rb") as f:
            return pickle.load(f)
    return None


def update_csv():
    rows = []
    for strat in STRATEGIES:
        ckpt = load_checkpoint(strat)
        if ckpt is None:
            continue
        mat = ckpt["acc_matrix"]
        per_task = [np.nanmean(mat[t]) for t in range(mat.shape[0])]
        rows.append({
            "strategie": strat,
            "avg_acc_final": float(np.nanmean(mat[-1])),
            "avg_acc_peak": float(np.nanmax(per_task)),
            "n_tasks": int(mat.shape[0]),
            "duration_s": round(ckpt["duration"], 1),
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S",
                                       time.localtime(ckpt["timestamp"])),
        })
    if rows:
        df = pd.DataFrame(rows)
        df.to_csv(CSV_FILE, index=False)
        return df
    return None


# --- Exécution ---
results = {}
state = load_state()
print(f"État précédent : {len(state)} stratégie(s) déjà terminée(s)")
for s, info in state.items():
    print(f"  ✓ {s} (durée : {info['duration']:.1f}s)")
print()

for strat in STRATEGIES:
    print("\n" + "=" * 80)
    print(f"STRATÉGIE : {strat}")
    print("=" * 80)

    ckpt = load_checkpoint(strat)
    if ckpt is not None:
        print(f"  ⏩ Reprise : résultat déjà calculé "
              f"(durée originale : {ckpt['duration']:.1f}s)")
        results[strat] = {
            "acc_matrix": ckpt["acc_matrix"],
            "duration": ckpt["duration"],
            "strategy_state": ckpt["strategy_state"],
        }
        continue

    try:
        model, strategy, acc_matrix, duration = run_strategy(
            strat, n_epochs=10, batch_size=32, lr=1e-3, seed=0,
        )
        results[strat] = {
            "acc_matrix": acc_matrix,
            "duration": duration,
            "strategy_state": get_strategy_state(strategy),
        }
        save_checkpoint(strat, acc_matrix, duration,
                        results[strat]["strategy_state"])
        print(f"✓ {strat} terminé en {duration:.1f}s")

    except KeyboardInterrupt:
        print(f"\n Interruption manuelle sur {strat}")
        break

    except Exception as e:
        print(f"\n Erreur sur {strat} : {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
        break


# Résumé
print("\n" + "=" * 80)
print("RÉSUMÉ FINAL")
print("=" * 80)
df = update_csv()
if df is not None:
    print(df.to_string(index=False))
    print(f"\nCSV : {CSV_FILE}")

État précédent : 0 stratégie(s) déjà terminée(s)


STRATÉGIE : ewc


Using cache found in C:\Users\hp/.cache\torch\hub\facebookresearch_dinov2_main


10:56:05 | INFO    | src.models.backbone | Backbone tronqué : 10 blocs sur 12
10:56:05 | INFO    | src.models.backbone | Backbone gelé


[Tâche 0] époques:   0%|          | 0/10 [00:00<?, ?it/s]

  epoch 1/10:   0%|          | 0/117 [00:00<?, ?it/s]

...

20:34:25 | INFO    | src.training.trainer | [Tâche 1] époque 9 | loss=0.0090 | train_acc=0.0000
  Tâche 1 | new=['Textile', 'Organique'] | new_acc=0.994 | old_acc=0.883 | overall=0.939 | t=20058s


[Tâche 2] époques:   0%|          | 0/10 [00:00<?, ?it/s]

  epoch 1/10:   0%|          | 0/40 [00:00<?, ?it/s]

20:43:52 | INFO    | src.training.trainer | [Tâche 2] époque 0 | loss=1.0596 | train_acc=0.0941


  epoch 2/10:   0%|          | 0/40 [00:00<?, ?it/s]

## 8. Matrice d'accuracy et analyse

In [ ]:
for strat in STRATEGIES:
    if strat not in results:
        continue
    mat = results[strat]["acc_matrix"]
    print(f"\n{strat.upper()} — Matrice d'accuracy :")
    df_mat = pd.DataFrame(mat,
                           index=[f"Après T{i}" for i in range(mat.shape[0])],
                           columns=name_label)
    print(df_mat.round(3).to_string())

    # Oubli par classe
    print(f"\n  Oubli par classe :")
    for c in range(n_classes):
        col = mat[:, c]
        valid = ~np.isnan(col)
        if valid.sum() >= 2:
            peak = np.nanmax(col)
            final = col[-1]
            drop = peak - final
            flag = "🔴" if drop > 0.20 else "🟢"
            print(f"    {flag} {name_label[c]:12s} : peak={peak:.3f} → final={final:.3f} (drop={drop:.3f})")

## 9. Figures

In [ ]:
colors = {
    "replay":      "#0072B2",
    "ewc":         "#D55E00",
    "dual_memory": "#CC79A7",
}

fig, ax = plt.subplots(figsize=(8, 5))

for strat in STRATEGIES:
    if strat not in results:
        continue
    mat = results[strat]["acc_matrix"]
    per_task = []
    for t in range(mat.shape[0]):
        valid = ~np.isnan(mat[t])
        per_task.append(np.nanmean(mat[t][valid]) if valid.sum() > 0 else np.nan)

    ax.plot(range(len(per_task)), per_task,
            marker="o", markersize=9, linewidth=2.5,
            color=colors[strat], label=strat,
            markerfacecolor="white", markeredgewidth=2)
    for t, acc in enumerate(per_task):
        if not np.isnan(acc):
            ax.annotate(f"{acc:.3f}", xy=(t, acc), xytext=(0, 10),
                        textcoords="offset points", ha="center",
                        fontsize=9, color=colors[strat], fontweight="bold")

ax.set_xlabel("Tâche")
ax.set_ylabel("Accuracy moyenne sur classes vues")
ax.set_title("Évolution de l'accuracy — 5 classes de déchets", loc="left")
ax.set_xticks(range(len(tasks)))
ax.set_xticklabels([f"T{t['idx']}\n({len(t['classes'])} cls)" for t in tasks])
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left", frameon=False)
ax.grid(alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

out_dir = ROOT / "results"
out_dir.mkdir(parents=True, exist_ok=True)
fig.tight_layout()
fig.savefig(out_dir / "benchmark_waste5_accuracy.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)

for ax, strat in zip(axes, STRATEGIES):
    if strat not in results:
        ax.axis("off")
        continue
    mat = results[strat]["acc_matrix"]
    drops = []
    for c in range(n_classes):
        col = mat[:, c]
        valid = ~np.isnan(col)
        if valid.sum() >= 2:
            drops.append(np.nanmax(col) - col[-1])
        else:
            drops.append(0.0)

    x = np.arange(n_classes)
    colors_bar = ["#D55E00" if d > 0.20 else "#0072B2" for d in drops]
    ax.bar(x, drops, color=colors_bar, edgecolor="white", linewidth=0.8)
    ax.axhline(0.20, color="#D55E00", linestyle="--", linewidth=1, label="Seuil 20%")
    ax.set_title(strat, loc="left")
    ax.set_xticks(x)
    ax.set_xticklabels(name_label, rotation=45, ha="right")
    if ax is axes[0]:
        ax.set_ylabel("Oubli (peak - final)")
    ax.legend(frameon=False, fontsize=8)
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("Oubli par classe — rouge > 20%", fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(out_dir / "benchmark_waste5_forgetting.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Synthèse

In [ ]:
rows = []
for strat in STRATEGIES:
    if strat not in results:
        continue
    mat = results[strat]["acc_matrix"]
    drops = []
    for c in range(mat.shape[1]):
        col = mat[:, c]
        valid = ~np.isnan(col)
        if valid.sum() >= 2:
            drops.append(np.nanmax(col) - col[-1])
    mean_forget = float(np.mean(drops)) if drops else 0.0
    rows.append({
        "Stratégie": strat,
        "Avg acc finale": round(float(np.nanmean(mat[-1])), 4),
        "Oubli moyen": round(mean_forget, 4),
        "Durée (s)": round(results[strat]["duration"], 1),
    })

df = pd.DataFrame(rows)
print("\n" + "=" * 70)
print("TABLEAU RÉCAPITULATIF — 5 classes")
print("=" * 70)
print(df.to_string(index=False))
print("=" * 70)

df.to_csv(out_dir / "benchmark_waste5_summary.csv", index=False)
print(f"\nSauvegardé : {out_dir / 'benchmark_waste5_summary.csv'}")